In [ ]:
!pip install -q tensorflow tensorflow-hub opencv-python google-genai


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

from tensorflow.keras.models import load_model
from google.colab import userdata
from google import genai


In [ ]:
movenet = hub.load(
    "https://tfhub.dev/google/movenet/singlepose/thunder/4"
)
movenet_model = movenet.signatures["serving_default"]


In [ ]:
def run_movenet(frame_rgb):
    """
    Input: RGB frame (H, W, 3)
    Output: keypoints (17, 2)
    """
    img = tf.image.resize_with_pad(
        tf.expand_dims(frame_rgb, axis=0),
        256, 256
    )
    img = tf.cast(img, tf.int32)

    outputs = movenet_model(img)
    keypoints = outputs["output_0"].numpy()[0, 0, :, :2]

    return keypoints




In [ ]:
def extract_keypoints_from_video(video_path, target_frames=50):
    keypoints_sequence = []

    cap = cv2.VideoCapture(video_path)

    while len(keypoints_sequence) < target_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        keypoints = run_movenet(frame_rgb)

        keypoints_sequence.append(keypoints.flatten())  # (34,)

    cap.release()

    # Pad if video is shorter than 50 frames
    if len(keypoints_sequence) == 0:
        raise ValueError("No frames read from video.")

    while len(keypoints_sequence) < target_frames:
        keypoints_sequence.append(keypoints_sequence[-1])

    return np.array(keypoints_sequence)  # (50, 34)


In [ ]:
MODEL_PATH = "1D_CNN_LSTM_multi_output.keras"
model = load_model(MODEL_PATH)




In [ ]:
VIDEO_PATH = "test_video.mp4"

X = extract_keypoints_from_video(VIDEO_PATH)
print("Keypoints shape:", X.shape)  # MUST be (50, 34)

X = X.reshape(1, 50, 34)

style_pred, posture_pred = model.predict(X)

style_idx = np.argmax(style_pred)
posture_idx = np.argmax(posture_pred)


In [ ]:
STYLE_LABELS = ["Back", "Front", "Goblet"]
POSTURE_LABELS = [
    "Correct",
    "Insufficient_depth",
    "Rounded_back",
    "Weight_on_toes"
]

predicted_style = STYLE_LABELS[style_idx]
predicted_posture = POSTURE_LABELS[posture_idx]

In [ ]:
STYLE_FEEDBACK_TEMPLATES = {
    "Back": "You are performing a back squat.",
    "Front": "You are performing a front squat.",
    "Goblet": "You are performing a goblet squat."
}

POSTURE_FEEDBACK_TEMPLATES = {
    "Correct": (
        "Your squat technique is correct."
    ),
    "Insufficient_depth": (
        "You are not reaching sufficient squat depth. "
    ),
    "Rounded_back": (
        "Your lower back is rounding. Keep a neutral spine."
    ),
    "Weight_on_toes": (
        "Your weight is shifting toward your toes. Keep pressure on your heels."
    )
}

def build_feedback_template(style, posture):
    return STYLE_FEEDBACK_TEMPLATES[style] + " " + POSTURE_FEEDBACK_TEMPLATES[posture]

template_text = build_feedback_template(predicted_style, predicted_posture)


In [ ]:
api_key = userdata.get("GOOGLE")
client = genai.Client(api_key=api_key)

In [ ]:
def rephrase_feedback_with_gemini(template_text):
    prompt = (
        "Rephrase the following fitness coaching message into a natural, "
        "clear, and encouraging sentence. Do not add or remove advice.\n\n"
        f'Message: "{template_text}"'
    )

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text.strip()


In [ ]:
final_feedback = rephrase_feedback_with_gemini(template_text)

print("\nFinal Gemini feedback:")
print(final_feedback)